In [2]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama

C:\Users\Ojas Pal\.conda\envs\learninggenai\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\Ojas Pal\AppData\Local\Temp\ipykernel_1676\2456446998.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


In [3]:
def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)

    documents = loader.load()
    return documents

In [4]:
extracted_data = load_pdf("data/")

In [6]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [7]:
text_chunks = text_split(extracted_data)
len(text_chunks)

5860

In [8]:
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    return embeddings

In [9]:
embeddings = download_hugging_face_embeddings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
embeddings

HuggingFaceEmbeddings(model_name='BAAI/bge-small-en-v1.5', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [11]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [12]:
load_dotenv()

True

In [13]:
import os

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)

In [14]:
index_name = "medical-chatbot"

In [15]:
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name = index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [16]:
docsearch = PineconeVectorStore.from_texts(
    texts= [t.page_content for t in text_chunks],
    embedding=embeddings,
    index_name=index_name
)

In [17]:
query = "What are allergies"

docs = docsearch.similarity_search(query, k=3)
print(docs)

[Document(id='e19508c6-1707-485d-a5c0-4d123645eceb', metadata={}, page_content='Purpose\nAllergy is a reaction of the immune system. Nor-\nmally, the immune system responds to foreign microor-\nganisms and particles, like pollen or dust, by producing\nspecific proteins called antibodies that are capable of\nbinding to identifying molecules, or antigens, on the\nforeign organisms. This reaction between antibody and\nantigen sets off a series of reactions designed to protect\nthe body from infection. Sometimes, this same series of'), Document(id='0ffc87bb-7831-4a89-bb4b-e83f29b77e44', metadata={}, page_content='Purpose\nAllergy is a reaction of the immune system. Nor-\nmally, the immune system responds to foreign microor-\nganisms and particles, like pollen or dust, by producing\nspecific proteins called antibodies that are capable of\nbinding to identifying molecules, or antigens, on the\nforeign organisms. This reaction between antibody and\nantigen sets off a series of reactions desig

In [18]:
llm = ChatOllama(model="llama3.1:8b", temperature=0.5)

In [19]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful medical assistant. Use the following context to answer the user's question.\n"
        "If you don't know the answer, just say that you don't know. Do not make up an answer.\n\n"
        "Context:\n{context}"
    ),
    ("human", "{question}")
])

In [20]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [21]:
retriever = docsearch.as_retriever(search_kwargs={'k': 2})

In [23]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [27]:
while True:
    user_input = input("Input Prompt: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    response = rag_chain.invoke(user_input)
    print("Response : ", response)

Response :  If you're experiencing a high fever (38°–40°C), it's essential to take steps to manage it and seek medical attention if necessary. Here are some general guidelines:

1. Drink plenty of fluids, such as water, clear broths, or electrolyte-rich beverages to help replace lost fluids and electrolytes.
2. Take over-the-counter medications like acetaminophen (Tylenol) or ibuprofen (Advil, Motrin) to help reduce the fever. However, always follow the recommended dosage instructions.
3. Rest and try to stay cool, as high temperatures can make you feel even more uncomfortable.
4. If the fever is accompanied by other symptoms like muscle aches, headache, chills, or loss of appetite, it's a good idea to consult with a healthcare professional.

However, if you suspect you or someone else has lymphangitis (based on the symptoms described), it's crucial to call your doctor immediately or visit an emergency room. A high fever and painful, red streaks just below the skin surface are diagnost